# Phase 1 / Step 1 - SOLD Data Exploration and Verification

Run:  `python notebooks/01_data_exploration.py`

(or paste cell by cell into a Colab notebook)

Prints a report answering every question you must settle before writing
any model code. Read the OUTPUT, don't just run it.

## Setup

In [3]:
import sys
import os
from collections import Counter

# Handle both script and notebook contexts
if '__file__' in dir():
    sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
else:
    sys.path.insert(0, "../src")

import numpy as np
import pandas as pd

from data import load_sold

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)


def rule(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

## 1. Load

In [4]:
rule("1. LOAD")

train = load_sold("train")
test = load_sold("test")

print(f"train rows: {len(train):,}   (expected 7,500)")
print(f"test  rows: {len(test):,}   (expected 2,500)")
print(f"total     : {len(train) + len(test):,}   (expected 10,000)")
print("\ncolumns:", list(train.columns))
print("\ndtypes:")
print(train.dtypes)


1. LOAD
train rows: 7,500   (expected 7,500)
test  rows: 2,500   (expected 2,500)
total     : 10,000   (expected 10,000)

columns: ['post_id', 'text', 'tokens', 'rationales', 'label', 'token_list', 'n_tokens', 'rationales_raw', 'raw_empty', 'length_mismatch', 'n_offensive_tokens']

dtypes:
post_id                int64
text                  object
tokens                object
rationales            object
label                 object
token_list            object
n_tokens               int64
rationales_raw        object
raw_empty               bool
length_mismatch         bool
n_offensive_tokens     int64
dtype: object


## 2. Raw Column Types

What did rationales/tokens actually arrive as?

In [5]:
rule("2. RAW COLUMN TYPES")

print("rationale column found as:", train.attrs.get("source_rationale_column"))
print("(GitHub README says 'rationals'; the real dataset uses 'rationales')\n")
raw = train.iloc[0]
print("tokens          type:", type(raw["tokens"]).__name__)
print("rationales_raw  type:", type(raw["rationales_raw"]).__name__, "(after parsing)")
print("label           type:", type(raw["label"]).__name__)
print("\nfirst row, raw text field:")
print(repr(raw["text"])[:300])


2. RAW COLUMN TYPES
rationale column found as: rationales
(GitHub README says 'rationals'; the real dataset uses 'rationales')

tokens          type: str
rationales_raw  type: list (after parsing)
label           type: str

first row, raw text field:
'@USER @USER  පට්ට පට පට...'


## 3. Sentence-Level Labels

What are the actual label values?

In [6]:
rule("3. SENTENCE-LEVEL LABELS")

for name, df in [("train", train), ("test", test)]:
    counts = df["label"].value_counts()
    print(f"\n{name}:")
    for k, v in counts.items():
        print(f"  {k!r:<20} {v:>6,}  ({v / len(df):.1%})")

print("\nWhole dataset should be 4,191 offensive / 5,809 not offensive.")
combined = pd.concat([train["label"], test["label"]]).value_counts()
print(dict(combined))


3. SENTENCE-LEVEL LABELS

train:
  'NOT'                 4,324  (57.7%)
  'OFF'                 3,176  (42.3%)

test:
  'NOT'                 1,485  (59.4%)
  'OFF'                 1,015  (40.6%)

Whole dataset should be 4,191 offensive / 5,809 not offensive.
{'NOT': np.int64(5809), 'OFF': np.int64(4191)}


## 4. Empty Rationales and Length Alignment

**⚠️ CRITICAL** — Does every token have exactly one rationale?

In [7]:
rule("4. EMPTY RATIONALES AND LENGTH ALIGNMENT   <-- CRITICAL")

print("In SOLD, NOT-offensive tweets store [] rather than a vector of zeros.")
print("load_sold() expands [] to [0]*n_tokens. Counts below are BEFORE expansion.\n")

for name, df in [("train", train), ("test", test)]:
    print(f"{name}:")
    print(f"  rows with empty rationale []      : {df['raw_empty'].sum():,} "
          f"({df['raw_empty'].mean():.1%})")
    print(f"  rows with non-empty but WRONG len : {df['length_mismatch'].sum():,}   <-- real problem if > 0")
    bad = df[df["length_mismatch"]]
    for _, r in bad.head(5).iterrows():
        print(f"     post_id={r['post_id']} tokens={r['n_tokens']} rationales={len(r['rationales_raw'])}")
    ok = all(len(r) == n for r, n in zip(df["rationales"], df["n_tokens"]))
    print(f"  after expansion, all lengths match: {ok}")
    print()

print("Cross-tab of empty-rationale against sentence label:")
for name, df in [("train", train), ("test", test)]:
    print(f"\n  {name}:")
    print(pd.crosstab(df["label"], df["raw_empty"]).rename(
        columns={True: "empty []", False: "has vector"}).to_string())

print("""
EXPECTED: every NOT row is empty. Any OFF row that is ALSO empty is an
annotation anomaly - the tweet was judged offensive but no tokens were
highlighted. Count them, note the number in the README, and decide whether
to keep them (they become all-negative training rows) or drop them.
DECISION REQUIRED - write it down either way.
""")


4. EMPTY RATIONALES AND LENGTH ALIGNMENT   <-- CRITICAL
In SOLD, NOT-offensive tweets store [] rather than a vector of zeros.
load_sold() expands [] to [0]*n_tokens. Counts below are BEFORE expansion.

train:
  rows with empty rationale []      : 4,523 (60.3%)
  rows with non-empty but WRONG len : 0   <-- real problem if > 0
  after expansion, all lengths match: True

test:
  rows with empty rationale []      : 1,549 (62.0%)
  rows with non-empty but WRONG len : 0   <-- real problem if > 0
  after expansion, all lengths match: True

Cross-tab of empty-rationale against sentence label:

  train:
raw_empty  has vector  empty []
label                          
NOT                 0      4324
OFF              2977       199

  test:
raw_empty  has vector  empty []
label                          
NOT                 0      1485
OFF               951        64

EXPECTED: every NOT row is empty. Any OFF row that is ALSO empty is an
annotation anomaly - the tweet was judged offensive but no t

## 5. Rationale Values

Are they strictly 0/1?

In [8]:
rule("5. RATIONALE VALUE SET")

vals = Counter()
for r in pd.concat([train["rationales_raw"], test["rationales_raw"]]):
    vals.update(r)
print("distinct values found:", dict(vals))
print("(expect only 0 and 1)")


5. RATIONALE VALUE SET
distinct values found: {0: 92600, 1: 9562}
(expect only 0 and 1)


## 6. The Scheme Check

Do non-offensive tweets have all-zero rationales?

In [9]:
rule("6. OFFENSIVE TWEETS SHOULD HAVE AT LEAST ONE OFFENSIVE TOKEN")

for name, df in [("train", train), ("test", test)]:
    labels = sorted(df["label"].unique())
    print(f"\n{name}:")
    for lab in labels:
        sub = df[df["label"] == lab]
        with_pos = (sub["n_offensive_tokens"] > 0).sum()
        print(f"  label {lab!r:<20} rows={len(sub):>5,}  "
              f"rows with >=1 offensive token: {with_pos:>5,} ({with_pos/len(sub):.1%})")

print("\nNOT rows must be 0.0% by construction (they store []).")
print("OFF rows below 100% are the anomaly from section 4. A small number is")
print("normal annotation noise; a large number means something is wrong.")


6. OFFENSIVE TWEETS SHOULD HAVE AT LEAST ONE OFFENSIVE TOKEN

train:
  label 'NOT'                rows=4,324  rows with >=1 offensive token:     0 (0.0%)
  label 'OFF'                rows=3,176  rows with >=1 offensive token: 2,967 (93.4%)

test:
  label 'NOT'                rows=1,485  rows with >=1 offensive token:     0 (0.0%)
  label 'OFF'                rows=1,015  rows with >=1 offensive token:   941 (92.7%)

NOT rows must be 0.0% by construction (they store []).
OFF rows below 100% are the anomaly from section 4. A small number is
normal annotation noise; a large number means something is wrong.


## 7. Class Imbalance

The number that drives the whole project

In [10]:
rule("7. TOKEN-LEVEL CLASS IMBALANCE")

for name, df in [("train", train), ("test", test)]:
    total_tokens = df["n_tokens"].sum()
    pos_tokens = df["n_offensive_tokens"].sum()
    print(f"{name}: {pos_tokens:,} offensive / {total_tokens:,} tokens "
          f"= {pos_tokens / total_tokens:.2%} positive")

print("\nPaper's 'All OFF' baseline has precision 0.03, so expect ~3%.")
print("If you see this, your rationale parsing is correct.")

# imbalance inside offensive tweets only
off_only = train[train["n_offensive_tokens"] > 0]
if len(off_only):
    print(f"\nWithin tweets that have >=1 offensive token (train, n={len(off_only):,}):")
    print(f"  {off_only['n_offensive_tokens'].sum() / off_only['n_tokens'].sum():.2%} of tokens are offensive")
    print(f"  median offensive tokens per such tweet: "
          f"{off_only['n_offensive_tokens'].median():.0f}")


7. TOKEN-LEVEL CLASS IMBALANCE
train: 7,294 offensive / 176,370 tokens = 4.14% positive
test: 2,268 offensive / 59,670 tokens = 3.80% positive

Paper's 'All OFF' baseline has precision 0.03, so expect ~3%.
If you see this, your rationale parsing is correct.

Within tweets that have >=1 offensive token (train, n=2,967):
  9.46% of tokens are offensive
  median offensive tokens per such tweet: 2


## 8. Length Distribution

Sets `max_len` for the model

In [11]:
rule("8. TOKEN COUNT DISTRIBUTION")

for name, df in [("train", train), ("test", test)]:
    n = df["n_tokens"]
    print(f"\n{name}: min={n.min()}  median={n.median():.0f}  mean={n.mean():.1f}  max={n.max()}")
    for p in [50, 75, 90, 95, 99]:
        print(f"  p{p}: {np.percentile(n, p):.0f}")

cover = {L: (train["n_tokens"] <= L).mean() for L in (16, 24, 32, 48, 64)}
print("\ncoverage if max_len set to:")
for L, c in cover.items():
    print(f"  {L:>3}: {c:.2%} of train tweets fit without truncation")
print("\nPick the smallest max_len covering >=99% and record it in configs/baseline.yaml.")


8. TOKEN COUNT DISTRIBUTION

train: min=5  median=19  mean=23.5  max=123
  p50: 19
  p75: 31
  p90: 49
  p95: 55
  p99: 79

test: min=5  median=20  mean=23.9  max=134
  p50: 20
  p75: 31
  p90: 49
  p95: 55
  p99: 84

coverage if max_len set to:
   16: 43.96% of train tweets fit without truncation
   24: 64.33% of train tweets fit without truncation
   32: 76.61% of train tweets fit without truncation
   48: 89.88% of train tweets fit without truncation
   64: 97.88% of train tweets fit without truncation

Pick the smallest max_len covering >=99% and record it in configs/baseline.yaml.


## 9. Vocabulary

First look, feeds the OOV analysis in Step 3

In [12]:
rule("9. VOCABULARY")

train_vocab = Counter(t for toks in train["token_list"] for t in toks)
test_vocab = Counter(t for toks in test["token_list"] for t in toks)
unseen = set(test_vocab) - set(train_vocab)
unseen_tok = sum(test_vocab[t] for t in unseen)

print(f"train vocab size: {len(train_vocab):,}")
print(f"test  vocab size: {len(test_vocab):,}")
print(f"test types unseen in train: {len(unseen):,} ({len(unseen)/len(test_vocab):.1%})")
print(f"test tokens unseen in train: {unseen_tok:,} "
      f"({unseen_tok / sum(test_vocab.values()):.1%} of all test tokens)")
print("\nThis last percentage is the empirical case for subword modelling.")
print("Record it - it goes in the paper.")

print("\nmost common tokens:")
for tok, c in train_vocab.most_common(15):
    print(f"  {c:>6,}  {tok}")


9. VOCABULARY
train vocab size: 33,004
test  vocab size: 15,751
test types unseen in train: 7,165 (45.5%)
test tokens unseen in train: 7,851 (13.2% of all test tokens)

This last percentage is the empirical case for subword modelling.
Record it - it goes in the paper.

most common tokens:
  19,439  .
   6,518  @USER
   2,307  ,
   1,589  #
   1,386  ?
   1,233  මේ
   1,120  එක
   1,116  නෑ
     988  !
     905  "
     899  කියලා
     805  වගේ
     793  -
     754  URL
     717  ඒ


## 10. Tokens Most Often Labelled Offensive

Sanity check that labels mean something

In [13]:
rule("10. TOKENS MOST OFTEN LABELLED OFFENSIVE")

tok_total, tok_pos = Counter(), Counter()
for toks, rats in zip(train["token_list"], train["rationales"]):
    for t, r in zip(toks, rats):
        tok_total[t] += 1
        if r == 1:
            tok_pos[t] += 1

rows = [(t, tok_pos[t], tok_total[t], tok_pos[t] / tok_total[t])
        for t in tok_pos if tok_total[t] >= 20]
rows.sort(key=lambda x: (-x[3], -x[1]))

print(f"{'token':<20} {'off':>6} {'total':>7} {'rate':>7}")
for t, p, n, rate in rows[:20]:
    print(f"{t:<20} {p:>6} {n:>7} {rate:>6.0%}")

print("\nNOTE the rates. The paper shows keyword offensiveness mostly 30-50%,")
print("i.e. context decides. That is why a lexicon lookup cannot solve this")
print("and why you need a sequence model.")


10. TOKENS MOST OFTEN LABELLED OFFENSIVE
token                   off   total    rate
හුත්ත                    73      73   100%
කැරි                     43      43   100%
හුත්තෝ                   34      34   100%
හුත්තො                   28      28   100%
පොන්නයා                  27      27   100%
වේසිගෙ                   25      25   100%
පොන්න                   122     124    98%
පුකේ                     34      35    97%
වේසි                     75      78    96%
හුජ්ජ                   121     126    96%
වේස                      68      71    96%
පුක                     189     198    95%
පඩ                       21      22    95%
බිජ්ජ                    81      85    95%
පරයා                     20      21    95%
පකෝ                     108     114    95%
කිම්බ                    35      37    95%
සක්කිලි                 137     146    94%
පර                       49      53    92%
ගොබ්බ                    20      22    91%

NOTE the rates. The paper shows keyword offensiveness

## 11. Examples

Read these as a team (token / label)

In [14]:
rule("11. TWENTY EXAMPLES (token / label)")

sample = pd.concat([
    train[train["n_offensive_tokens"] > 0].head(12),
    train[train["n_offensive_tokens"] == 0].head(8),
])

for i, (_, r) in enumerate(sample.iterrows(), 1):
    print(f"\n--- {i}. post_id={r['post_id']}  label={r['label']}  "
          f"({r['n_offensive_tokens']}/{r['n_tokens']} offensive)")
    for t, lab in zip(r["token_list"], r["rationales"]):
        mark = "  <== OFFENSIVE" if lab == 1 else ""
        print(f"    {lab}  {t}{mark}")


11. TWENTY EXAMPLES (token / label)

--- 1. post_id=915618589855617026  label=OFF  (1/27 offensive)
    0  පරණ
    0  කෑල්ල
    0  අද
    0  වෙනකම්
    0  හිටියනම්
    0  අදට
    0  අවුරුදු
    0  4යි
    0  .
    0  යාලු
    0  වෙලා
    0  .
    0  වෙලාවට
    0  ඊට
    0  පහු
    0  වෙනිදම
    0  බූට්
    0  එක
    0  කෑවෙ
    0  ,
    0  නැත්තම්
    0  අවුරුදු
    0  4ක්
    0  එක
    0  කෙල්ලෙක්
    0  ?
    1  ටෞකනවා  <== OFFENSIVE

--- 2. post_id=1397219745707986955  label=OFF  (3/52 offensive)
    0  හොඳ
    0  මිතුරියක්
    0  කතා
    0  කලා
    0  .
    0  විස්තර
    0  කතාකරමින්
    0  ඉදලා
    0  මේ
    0  දවස්වල
    0  මොකො
    0  කරන්නේ
    0  ඇහුවා
    0  .
    0  මම
    0  උයපු
    0  හදපු
    0  කෑම
    0  බීම
    0  ජාති
    0  කිව්වා
    0  .
    0  බැඳලත්
    0  නැති
    0  එකෙ
    0  ඔච්චර
    0  උයන්න
    0  යන්න
    0  එපා
    1  තෝ  <== OFFENSIVE
    1  ගෑනියෙක්  <== OFFENSIVE
    0  වෙයි
    0  කියලා
    0  එයා
    0  කිව්ව.
    0  ඒක
    0  ඇහු
    0  වෙලේ
    

## 12. Summary for the README

Copy these numbers into your README

In [18]:
rule("12. COPY THESE NUMBERS INTO YOUR README")

tt_train = train["n_tokens"].sum()
tp_train = train["n_offensive_tokens"].sum()
tt_test = test["n_tokens"].sum()
tp_test = test["n_offensive_tokens"].sum()

print(f"""
train rows                 {len(train):,}
test rows                  {len(test):,}
empty rationale rows (tr)  {train['raw_empty'].sum():,}
length-mismatch rows (tr)  {train['length_mismatch'].sum()}
length-mismatch rows (te)  {test['length_mismatch'].sum()}
OFF w/ empty rationale []  {((train['label']=='OFF') & train['raw_empty']).sum():,} train / {((test['label']=='OFF') & test['raw_empty']).sum():,} test
OFF w/ no offensive token  {((train['label']=='OFF') & (train['n_offensive_tokens']==0)).sum():,} train / {((test['label']=='OFF') & (test['n_offensive_tokens']==0)).sum():,} test
train tokens               {tt_train:,}
train offensive tokens     {tp_train:,}  ({tp_train/tt_train:.2%})
test tokens                {tt_test:,}
test offensive tokens      {tp_test:,}  ({tp_test/tt_test:.2%})
train vocab                {len(train_vocab):,}
test tokens unseen in train {unseen_tok/sum(test_vocab.values()):.1%}
median tweet length        {train['n_tokens'].median():.0f} tokens
p99 tweet length           {np.percentile(train['n_tokens'], 99):.0f} tokens
""")


12. COPY THESE NUMBERS INTO YOUR README

train rows                 7,500
test rows                  2,500
empty rationale rows (tr)  4,523
length-mismatch rows (tr)  0
length-mismatch rows (te)  0
OFF w/ empty rationale []  199 train / 64 test
OFF w/ no offensive token  209 train / 74 test
train tokens               176,370
train offensive tokens     7,294  (4.14%)
test tokens                59,670
test offensive tokens      2,268  (3.80%)
train vocab                33,004
test tokens unseen in train 13.2%
median tweet length        19 tokens
p99 tweet length           79 tokens

